In [33]:
import hashlib
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from tqdm import tqdm
import os
import time

def create_session():
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=0.3, status_forcelist=[500, 502, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def get_pin_hash(pin):
    pin_str = f"{pin:08d}"
    return hashlib.sha1(pin_str.encode('utf-8')).hexdigest().upper()

def check_pins(session, pins):
    results = {}
    for pin in pins:
        pin_hash = get_pin_hash(pin)
        prefix = pin_hash[:5]
        suffix = pin_hash[5:]
        
        url = f"https://api.pwnedpasswords.com/range/{prefix}"
        
        try:
            response = session.get(url)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Request error encountered: {e}")
            time.sleep(10)
            continue
        
        hashes = (line.split(':') for line in response.text.splitlines())
        for h, count in hashes:
            if h == suffix:
                results[f"{pin:08d}"] = int(count)
                break
    
    return results

def save_checkpoint(pin, results):
    with open('checkpoint.txt', 'w') as f:
        f.write(f"{pin}\n")
        for pin, count in results.items():
            f.write(f"{pin}:{count}\n")

def load_checkpoint():
    if os.path.exists('checkpoint.txt'):
        with open('checkpoint.txt', 'r') as f:
            lines = f.readlines()
            last_pin = int(lines[0].strip())
            results = {}
            for line in lines[1:]:
                pin, count = line.strip().split(':')
                results[pin] = int(count)
            return last_pin, results
    return 0, {}

def main():
    start_pin, results = load_checkpoint()
    session = create_session()

    for batch_start in tqdm(range(start_pin, 100000000, 100)):
        batch_end = min(batch_start + 100, 100000000)
        batch_results = check_pins(session, range(batch_start, batch_end))
        results.update(batch_results)
        save_checkpoint(batch_end, results)
        time.sleep(1)  # Sleep for 1 second between batches

    # Sort and save results
    sorted_by_pin = sorted(results.items(), key=lambda x: x[0])
    sorted_by_count = sorted(results.items(), key=lambda x: x[1], reverse=True)

    with open('all_8digit_pins_with_counts.txt', 'w') as f:
        for pin, count in sorted_by_pin:
            f.write(f"{pin} : {count}\n")

    with open('8digit_pins_sorted_by_count.txt', 'w') as f:
        for pin, _ in sorted_by_count:
            f.write(f"{pin}\n")

    print("Scan complete. Results have been written to 'all_6digit_pins_with_counts.txt' and '6digit_pins_sorted_by_count.txt'.")

if __name__ == "__main__":
    main()

0it [00:00, ?it/s]


Scan complete. Results have been written to 'all_6digit_pins_with_counts.txt' and '6digit_pins_sorted_by_count.txt'.


In [29]:
import os

def add_missing_pins():
    # Read the current checkpoint file
    if not os.path.exists('checkpoint.txt'):
        print("Error: checkpoint.txt file not found!")
        return
    
    # Load the checkpoint data
    with open('checkpoint.txt', 'r') as f:
        lines = f.readlines()
    
    if len(lines) < 1:
        print("Error: checkpoint.txt appears to be empty or malformed!")
        return
    
    # Get the last checked PIN from the first line
    last_pin = int(lines[0].strip())
    
    # Parse existing results into a dictionary
    existing_pins = {}
    for line in lines[1:]:
        if ':' in line:
            pin, count = line.strip().split(':')
            existing_pins[pin] = int(count)
    
    print(f"Loaded {len(existing_pins)} pins from checkpoint.txt")
    print(f"Last checked PIN was: {last_pin}")
    
    # Create a complete set of all 8-digit PINs
    all_pins = {}
    for pin in range(0, 100000000):
        pin_str = f"{pin:08d}"
        # If the PIN exists in our data, keep its count, otherwise set to 0
        all_pins[pin_str] = existing_pins.get(pin_str, 0)
    
    print(f"Total number of 8-digit PINs: {len(all_pins)}")
    print(f"Adding {len(all_pins) - len(existing_pins)} missing PINs with count 0")
    
    # Write the updated checkpoint file
    with open('checkpoint_complete.txt', 'w') as f:
        # Write the last checked PIN
        f.write(f"{last_pin}\n")
        
        # Write all PINs with their counts
        for pin, count in sorted(all_pins.items()):
            f.write(f"{pin}:{count}\n")
    
    print("Updated checkpoint saved to checkpoint_complete.txt")
    print("Please rename this file to checkpoint.txt if you want to use it for further processing")

if __name__ == "__main__":
    add_missing_pins()

Loaded 5397489 pins from checkpoint.txt
Last checked PIN was: 100000000
Total number of 8-digit PINs: 100000000
Adding 94602511 missing PINs with count 0
Updated checkpoint saved to checkpoint_complete.txt
Please rename this file to checkpoint.txt if you want to use it for further processing
